# 🚀 Azure SQL Advisor — Recommendation Engine

**Notebook 2 of 2** — Reads telemetry from Azure Storage Account (ADLS Gen2) and generates prioritized recommendations.

This notebook:
1. Reads partitioned telemetry data from ADLS Gen2 for a configurable lookback window.
2. Runs 4 independent analyzers: **Performance**, **Index**, **Storage**, and **Cost**.
3. Scores and ranks all recommendations by priority.
4. Computes a **Health Score** (0–100) and executive summary.
5. Persists recommendations as partitioned Parquet files in ADLS Gen2 (`recommendations/...`).
6. Generates an interactive **HTML report**.


In [ ]:
# Databricks notebook source
# Create interactive widgets
dbutils.widgets.text("server_name", "", "Azure SQL Server FQDN")
dbutils.widgets.text("database_name", "", "Database Name")
dbutils.widgets.text("storage_account", "", "Storage Account Name")
dbutils.widgets.text("storage_container", "azure-sql-telemetry", "Container Name")
dbutils.widgets.text("lookback_days", "7", "Lookback Window (days)")

server_name = dbutils.widgets.get("server_name")
database_name = dbutils.widgets.get("database_name")
storage_account = dbutils.widgets.get("storage_account")
storage_container = dbutils.widgets.get("storage_container")
lookback_days = int(dbutils.widgets.get("lookback_days"))

print(f"Server: {server_name}")
print(f"Database: {database_name}")
print(f"Storage: {storage_account}/{storage_container}")
print(f"Lookback: {lookback_days} days")


### 📦 Import Configuration


In [ ]:
import sys, os, json
from datetime import datetime, timezone, timedelta

repo_path = os.path.dirname(os.path.abspath(globals().get('__file__', '/Workspace/Repos/azure_sql_advisor')))
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

from config import (
    AdvisorConfig, ALL_METRICS,
    Category, Severity, Effort, Risk, Confidence, Recommendation,
    SEVERITY_SCORES, AZURE_SQL_PRICING, WAIT_CATEGORIES,
)

cfg = AdvisorConfig(
    server=server_name,
    database=database_name,
    storage_account_name=storage_account,
    storage_container=storage_container,
    lookback_days=lookback_days,
)

ABFSS_BASE = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net"
RAW_BASE = f"{ABFSS_BASE}/{cfg.storage_base_path}/{server_name}/{database_name}"
RECS_BASE = f"{ABFSS_BASE}/{cfg.recommendations_base_path}/{server_name}/{database_name}"
now = datetime.now(timezone.utc)

print(f"Raw Base: {RAW_BASE}")
print(f"Recommendations Base: {RECS_BASE}")


### 📂 Load Telemetry from Azure Storage Account

Read partitioned Parquet data from `raw/{server}/{database}/{metric}/year=YYYY/...` for the configured lookback window.


In [ ]:
from pyspark.sql.functions import col, lit, max as spark_max

def load_metric(metric_name: str, lookback: int = lookback_days) -> 'DataFrame':
    """Load a metric from ADLS Gen2 with optional date-based filtering."""
    base_path = f"{RAW_BASE}/{metric_name}"
    try:
        df = spark.read.parquet(f"{base_path}/")
        # Filter to lookback window if ingestion_time column exists
        if "ingestion_time" in df.columns:
            cutoff = now - timedelta(days=lookback)
            df = df.filter(col("ingestion_time") >= lit(cutoff))
        count = df.count()
        print(f"  ✅ {metric_name}: {count} records (lookback={lookback}d)")
        return df
    except Exception as e:
        print(f"  ⚠️ {metric_name}: not available ({e})")
        return spark.createDataFrame([], schema="dummy STRING")


print("Loading telemetry datasets...")
print("=" * 60)

datasets = {}
for metric in ALL_METRICS:
    datasets[metric] = load_metric(metric)

print("\nTelemetry loading complete.")


In [ ]:
# Convert PySpark DataFrames to Python list-of-dicts for analyzers
data = {"collection_errors": {}}

for metric_name, df in datasets.items():
    try:
        if df.columns == ["dummy"]:
            data[metric_name] = []
        else:
            data[metric_name] = [row.asDict() for row in df.collect()]
    except Exception as e:
        print(f"  Error converting {metric_name}: {e}")
        data[metric_name] = []

print(f"Converted {len(data)} metric datasets to Python dicts.")
for m, records in data.items():
    if m != 'collection_errors' and isinstance(records, list):
        print(f"  {m}: {len(records)} records")


## 🏎️ Performance Analyzer

Analyzes CPU/IO pressure, wait statistics, expensive queries, and Query Store regressions.


In [ ]:
def analyze_performance(data: dict, config) -> list:
    """Performance analyzer: CPU, IO, waits, expensive queries, Query Store."""
    recommendations = []
    
    # 1. Resource utilization (CPU)
    resource_stats = data.get('resource_stats', [])
    if resource_stats:
        cpu_values = [(r.get('avg_cpu_percent') or r.get('avg_cpu', 0) or 0) for r in resource_stats]
        avg_cpu = sum(cpu_values) / max(len(cpu_values), 1)
        
        if avg_cpu > config.cpu_critical_pct:
            recommendations.append(Recommendation(
                title="High CPU Utilization",
                category=Category.PERFORMANCE, severity=Severity.CRITICAL,
                description=f"Average CPU is very high ({avg_cpu:.1f}%) across {len(resource_stats)} samples.",
                impact_description="Scale up compute or optimize top queries to improve performance.",
                effort=Effort.MODERATE, risk=Risk.LOW, confidence=Confidence.HIGH,
                source_metric="resource_stats",
            ))
        elif avg_cpu > config.cpu_high_pct:
            recommendations.append(Recommendation(
                title="Elevated CPU Utilization",
                category=Category.PERFORMANCE, severity=Severity.HIGH,
                description=f"Average CPU is elevated ({avg_cpu:.1f}%) across {len(resource_stats)} samples.",
                impact_description="Optimizing top CPU-consuming queries can free up resources.",
                effort=Effort.MODERATE, risk=Risk.LOW, confidence=Confidence.MEDIUM,
                source_metric="resource_stats",
            ))
    
    # 2. Wait statistics
    wait_stats = data.get('wait_stats', [])
    if wait_stats:
        top_wait = sorted(wait_stats, key=lambda x: x.get('wait_time_ms', 0) or 0, reverse=True)
        if top_wait:
            wt = top_wait[0].get('wait_type', '')
            category = WAIT_CATEGORIES.get(wt)
            if category:
                recommendations.append(Recommendation(
                    title=f"High {category} Waits",
                    category=Category.PERFORMANCE, severity=Severity.HIGH,
                    description=f"Dominant wait type is {wt}, indicating a {category} bottleneck.",
                    impact_description=f"Addressing {category} bottlenecks will improve concurrency and response times.",
                    effort=Effort.SIGNIFICANT, risk=Risk.MEDIUM, confidence=Confidence.HIGH,
                    source_metric="wait_stats",
                ))
    
    # 3. Expensive queries (CPU)
    for q in data.get('top_queries_cpu', []):
        avg_cpu_ms = q.get('avg_cpu_ms', 0) or 0
        if avg_cpu_ms > config.query_duration_high_ms:
            recommendations.append(Recommendation(
                title="Expensive Query (CPU)",
                category=Category.PERFORMANCE, severity=Severity.CRITICAL,
                description=f"Query {q.get('query_id', q.get('query_hash', 'N/A'))} avg CPU: {avg_cpu_ms}ms.",
                impact_description="Optimizing this query will reduce overall CPU load.",
                effort=Effort.MODERATE, risk=Risk.MEDIUM, confidence=Confidence.HIGH,
                details={'query_text': q.get('query_text', '')},
                source_metric="top_queries_cpu",
            ))
    
    # 4. Expensive queries (IO)
    for q in data.get('top_queries_reads', []):
        avg_reads = q.get('avg_logical_reads', 0) or 0
        if avg_reads > 100_000:
            recommendations.append(Recommendation(
                title="Expensive Query (IO)",
                category=Category.PERFORMANCE, severity=Severity.HIGH,
                description=f"Query {q.get('query_id', q.get('query_hash', 'N/A'))} has {avg_reads} avg logical reads.",
                impact_description="Adding indexes or rewriting this query will reduce IO load.",
                effort=Effort.MODERATE, risk=Risk.MEDIUM, confidence=Confidence.HIGH,
                details={'query_text': q.get('query_text', '')},
                source_metric="top_queries_reads",
            ))
    
    # 5. Query Store
    qs_stats = data.get('query_store_stats', [])
    if not qs_stats:
        recommendations.append(Recommendation(
            title="Enable Query Store",
            category=Category.PERFORMANCE, severity=Severity.MEDIUM,
            description="Query Store data is empty — likely disabled.",
            impact_description="Enable Query Store for deep query regression insights.",
            action_sql="ALTER DATABASE CURRENT SET QUERY_STORE = ON;",
            effort=Effort.QUICK_WIN, risk=Risk.LOW, confidence=Confidence.HIGH,
            source_metric="query_store_stats",
        ))
    else:
        for qs in qs_stats:
            if qs.get('regressed', False):
                qid = qs.get('query_id')
                pid = qs.get('last_good_plan_id')
                recommendations.append(Recommendation(
                    title="Query Performance Regression",
                    category=Category.PERFORMANCE, severity=Severity.HIGH,
                    description=f"Query {qid} has regressed.",
                    impact_description="Force the last known good plan.",
                    action_sql=f"EXEC sp_query_store_force_plan @query_id = {qid}, @plan_id = {pid};" if pid else "",
                    effort=Effort.QUICK_WIN, risk=Risk.MEDIUM, confidence=Confidence.HIGH,
                    source_metric="query_store_stats",
                ))
    
    return recommendations

perf_recs = analyze_performance(data, cfg)
print(f"Performance recommendations: {len(perf_recs)}")


## 🗂️ Index Analyzer

Identifies missing indexes, unused indexes, duplicates, and high fragmentation.


In [ ]:
def analyze_indexes(data: dict, config) -> list:
    """Index analyzer: missing, unused, duplicate, and fragmented indexes."""
    recommendations = []
    
    # 1. Missing indexes
    for mi in data.get('missing_indexes', []):
        score = (mi.get('avg_user_impact') or mi.get('improvement_score', 0) or 0)
        if score > 80:   sev = Severity.CRITICAL
        elif score > 50: sev = Severity.HIGH
        elif score > 25: sev = Severity.MEDIUM
        else:            sev = Severity.LOW
        
        recommendations.append(Recommendation(
            title=f"Missing Index on {mi.get('table_name', 'Unknown')}",
            category=Category.INDEX, severity=sev,
            description=f"Creating this index could improve performance (impact: {score}%).",
            impact_description="Faster query execution for workloads filtering on these columns.",
            action_sql=mi.get('create_index_ddl', ''),
            effort=Effort.MODERATE, risk=Risk.MEDIUM, confidence=Confidence.HIGH,
            estimated_impact_pct=float(score),
            details={'columns': mi.get('equality_columns', mi.get('columns', ''))},
            source_metric="missing_indexes",
        ))
    
    # 2. Unused indexes
    for ui in data.get('unused_indexes', []):
        size_mb = ui.get('size_mb', 0)
        updates = ui.get('user_updates', 0)
        recommendations.append(Recommendation(
            title=f"Drop Unused Index {ui.get('index_name', 'Unknown')}",
            category=Category.INDEX, severity=Severity.MEDIUM,
            description=f"Index has {updates} updates but zero reads. Takes {size_mb}MB.",
            impact_description="Reclaims storage and reduces write overhead.",
            action_sql=ui.get('drop_index_ddl', ''),
            effort=Effort.QUICK_WIN, risk=Risk.MEDIUM, confidence=Confidence.HIGH,
            source_metric="unused_indexes",
        ))
    
    # 3. Duplicate indexes
    for di in data.get('duplicate_indexes', []):
        idx_a = di.get('index_a', di.get('index_name1', ''))
        idx_b = di.get('index_b', di.get('index_name2', ''))
        tbl = di.get('table_name', '')
        recommendations.append(Recommendation(
            title=f"Duplicate Index: {idx_a}",
            category=Category.INDEX, severity=Severity.MEDIUM,
            description=f"Indexes {idx_a} and {idx_b} share key columns.",
            impact_description="Removing duplicates reduces storage and write overhead.",
            action_sql=f"DROP INDEX [{idx_b}] ON [{tbl}];",
            effort=Effort.QUICK_WIN, risk=Risk.MEDIUM, confidence=Confidence.HIGH,
            source_metric="duplicate_indexes",
        ))
    
    # 4. Fragmented indexes
    for frag in data.get('index_fragmentation', []):
        pct = (frag.get('avg_fragmentation_in_percent') or frag.get('fragmentation_pct', 0) or 0)
        idx_name = frag.get('index_name', '')
        schema = frag.get('schema_name', 'dbo')
        table = frag.get('table_name', '')
        pages = frag.get('page_count', 0)
        action = frag.get('recommended_action')
        
        if pct > 30:
            recommendations.append(Recommendation(
                title=f"Rebuild Fragmented Index {idx_name}",
                category=Category.INDEX, severity=Severity.HIGH,
                description=f"Fragmentation is {pct:.1f}% ({pages} pages).",
                impact_description="Rebuilding restores sequential read performance.",
                action_sql=action or f"ALTER INDEX [{idx_name}] ON [{schema}].[{table}] REBUILD;",
                effort=Effort.MODERATE, risk=Risk.LOW, confidence=Confidence.HIGH,
                source_metric="index_fragmentation",
            ))
        elif pct > 10:
            recommendations.append(Recommendation(
                title=f"Reorganize Fragmented Index {idx_name}",
                category=Category.INDEX, severity=Severity.MEDIUM,
                description=f"Fragmentation is {pct:.1f}% ({pages} pages).",
                impact_description="Reorganizing reduces fragmentation with minimal locking.",
                action_sql=action or f"ALTER INDEX [{idx_name}] ON [{schema}].[{table}] REORGANIZE;",
                effort=Effort.QUICK_WIN, risk=Risk.LOW, confidence=Confidence.HIGH,
                source_metric="index_fragmentation",
            ))
    
    return recommendations

index_recs = analyze_indexes(data, cfg)
print(f"Index recommendations: {len(index_recs)}")


## 💾 Storage Analyzer

Audits large tables, compression opportunities, data type usage, and database file space.


In [ ]:
def analyze_storage(data: dict, config) -> list:
    """Storage analyzer: table sizes, compression, data types, file space."""
    recommendations = []
    
    # 1. Large tables
    for ts in data.get('table_sizes', []):
        size_gb = (ts.get('reserved_mb', 0) or 0) / 1024
        if size_gb > config.table_size_concern_gb:
            recommendations.append(Recommendation(
                title=f"Very Large Table: {ts.get('table_name', 'Unknown')}",
                category=Category.STORAGE, severity=Severity.MEDIUM,
                description=f"Table is {size_gb:.1f}GB (threshold: {config.table_size_concern_gb}GB).",
                impact_description="Consider partitioning or archival strategies.",
                effort=Effort.SIGNIFICANT, risk=Risk.MEDIUM, confidence=Confidence.HIGH,
                source_metric="table_sizes",
            ))
        
        # Unused space
        reserved = ts.get('reserved_mb', 0) or 0
        unused = ts.get('unused_mb', 0) or 0
        if reserved > 0 and (unused / reserved) > 0.1:
            recommendations.append(Recommendation(
                title=f"High Unused Space: {ts.get('table_name', 'Unknown')}",
                category=Category.STORAGE, severity=Severity.LOW,
                description=f"{unused:.0f}MB unused ({unused/reserved*100:.1f}% of total).",
                impact_description="Reclaiming unused space reduces storage costs.",
                effort=Effort.MODERATE, risk=Risk.LOW, confidence=Confidence.MEDIUM,
                source_metric="table_sizes",
            ))
    
    # 2. Compression opportunities
    for cc in data.get('compression_candidates', []):
        size_mb = cc.get('size_mb', 0) or 0
        schema = cc.get('schema_name', 'dbo')
        table = cc.get('table_name', '')
        if size_mb > 100:
            recommendations.append(Recommendation(
                title=f"PAGE Compression: {schema}.{table}",
                category=Category.STORAGE, severity=Severity.MEDIUM,
                description=f"Table is {size_mb}MB. PAGE compression can save ~60%.",
                impact_description="Significantly reduces storage costs and IO.",
                action_sql=f"ALTER TABLE [{schema}].[{table}] REBUILD WITH (DATA_COMPRESSION = PAGE);",
                effort=Effort.MODERATE, risk=Risk.LOW, confidence=Confidence.HIGH,
                estimated_impact_pct=60.0,
                source_metric="compression_candidates",
            ))
        else:
            recommendations.append(Recommendation(
                title=f"ROW Compression: {schema}.{table}",
                category=Category.STORAGE, severity=Severity.LOW,
                description=f"Table is {size_mb}MB. ROW compression can save ~30%.",
                impact_description="Reduces storage footprint with minimal CPU overhead.",
                action_sql=f"ALTER TABLE [{schema}].[{table}] REBUILD WITH (DATA_COMPRESSION = ROW);",
                effort=Effort.QUICK_WIN, risk=Risk.LOW, confidence=Confidence.HIGH,
                estimated_impact_pct=30.0,
                source_metric="compression_candidates",
            ))
    
    # 3. Data type optimization (NVARCHAR audit)
    for dt in data.get('data_type_audit', []):
        col_type = (dt.get('data_type') or dt.get('column_type', '')).upper()
        if col_type == 'NVARCHAR' and not dt.get('needs_unicode', False):
            recommendations.append(Recommendation(
                title=f"VARCHAR instead of NVARCHAR in {dt.get('table_name', '')}",
                category=Category.STORAGE, severity=Severity.MEDIUM,
                description=f"Column {dt.get('column_name', '')} is NVARCHAR — using VARCHAR saves 50%.",
                impact_description="Halves storage for this column.",
                effort=Effort.MODERATE, risk=Risk.MEDIUM, confidence=Confidence.MEDIUM,
                source_metric="data_type_audit",
            ))
    
    # 4. Database file space
    db_files = data.get('database_files', [])
    log_size = sum(f.get('size_mb', 0) or 0 for f in db_files if f.get('file_type', f.get('type_desc', '')) == 'LOG')
    data_size = sum(f.get('size_mb', 0) or 0 for f in db_files if f.get('file_type', f.get('type_desc', '')) == 'ROWS')
    if log_size > data_size and data_size > 0:
        recommendations.append(Recommendation(
            title="Log File Larger Than Data File",
            category=Category.STORAGE, severity=Severity.MEDIUM,
            description=f"Log: {log_size:.0f}MB vs Data: {data_size:.0f}MB.",
            impact_description="Check transaction log backup frequency.",
            effort=Effort.MODERATE, risk=Risk.LOW, confidence=Confidence.HIGH,
            source_metric="database_files",
        ))
    
    return recommendations

storage_recs = analyze_storage(data, cfg)
print(f"Storage recommendations: {len(storage_recs)}")


## 💰 Cost Analyzer

Evaluates service tier rightsizing, serverless opportunities, reserved capacity, and Azure Hybrid Benefit.


In [ ]:
def analyze_cost(data: dict, config) -> list:
    """Cost analyzer: tier rightsizing, serverless, reserved capacity, Hybrid Benefit."""
    recommendations = []
    
    resource_stats = data.get('resource_stats', [])
    service_tier = data.get('service_tier', [])
    tier_str = 'Unknown'
    if isinstance(service_tier, list) and service_tier:
        st = service_tier[0]
        if isinstance(st, dict):
            tier_str = f"{st.get('edition', '')} {st.get('service_objective', '')}".strip()
    
    if resource_stats:
        cpu_values = [(r.get('avg_cpu_percent') or r.get('avg_cpu', 0) or 0) for r in resource_stats]
        io_values = [(r.get('avg_data_io_percent') or r.get('avg_io', 0) or 0) for r in resource_stats]
        avg_cpu = sum(cpu_values) / max(len(cpu_values), 1)
        avg_io = sum(io_values) / max(len(io_values), 1)
        
        # 1. Scale down
        if avg_cpu < config.underutilized_cpu_pct and avg_io < config.underutilized_io_pct:
            recommendations.append(Recommendation(
                title="Scale Down Service Tier",
                category=Category.COST, severity=Severity.HIGH,
                description=f"Avg CPU ({avg_cpu:.1f}%) and IO ({avg_io:.1f}%) are underutilized.",
                impact_description="Immediate monthly cost savings without performance impact.",
                effort=Effort.MODERATE, risk=Risk.MEDIUM, confidence=Confidence.HIGH,
                details={'current_tier': tier_str, 'avg_cpu': avg_cpu, 'avg_io': avg_io},
                source_metric="resource_stats",
            ))
        
        # 2. Serverless
        idle_periods = [r for r in resource_stats if (r.get('avg_cpu_percent') or r.get('avg_cpu', 0) or 0) < 2]
        if len(idle_periods) > 1:
            recommendations.append(Recommendation(
                title="Consider Serverless Compute Tier",
                category=Category.COST, severity=Severity.MEDIUM,
                description="Significant idle periods detected. Serverless auto-pause could reduce costs.",
                impact_description="Per-second billing; auto-pause eliminates idle compute costs.",
                effort=Effort.MODERATE, risk=Risk.LOW, confidence=Confidence.MEDIUM,
                source_metric="resource_stats",
            ))
        
        # 3. Reserved capacity
        if avg_cpu > 30:
            recommendations.append(Recommendation(
                title="Purchase Reserved Capacity",
                category=Category.COST, severity=Severity.MEDIUM,
                description="Stable workload. Reserved capacity offers significant discounts.",
                impact_description="~33% savings (1-year) or ~55% savings (3-year).",
                effort=Effort.QUICK_WIN, risk=Risk.LOW, confidence=Confidence.HIGH,
                source_metric="resource_stats",
            ))
    
    # 4. Hybrid Benefit
    if not data.get('hybrid_benefit_enabled', False):
        recommendations.append(Recommendation(
            title="Enable Azure Hybrid Benefit",
            category=Category.COST, severity=Severity.INFO,
            description="Use existing SQL Server licenses with Software Assurance.",
            impact_description="Significantly reduces vCore-based database costs.",
            effort=Effort.QUICK_WIN, risk=Risk.LOW, confidence=Confidence.HIGH,
            source_metric="service_tier",
        ))
    
    return recommendations

cost_recs = analyze_cost(data, cfg)
print(f"Cost recommendations: {len(cost_recs)}")


## 🏆 Score, Rank & Build Executive Summary


In [ ]:
# Combine all recommendations
all_recs = perf_recs + index_recs + storage_recs + cost_recs
print(f"Total recommendations: {len(all_recs)}")

# Score and rank
scored = []
for rec in all_recs:
    base_score = SEVERITY_SCORES.get(rec.severity, 25)
    impact_bonus = rec.estimated_impact_pct * 0.3
    cat_weight = {
        Category.PERFORMANCE: cfg.weight_performance,
        Category.INDEX: cfg.weight_performance,
        Category.STORAGE: cfg.weight_storage,
        Category.COST: cfg.weight_cost,
    }.get(rec.category, 0.33)
    priority_score = min(100, (base_score + impact_bonus) * (0.5 + cat_weight))
    scored.append({
        'priority_score': round(priority_score, 1),
        'recommendation': rec,
    })

ranked = sorted(scored, key=lambda x: x['priority_score'], reverse=True)
for i, r in enumerate(ranked, 1):
    r['priority_rank'] = i

# Health Score
penalty = 0
for rec in all_recs:
    if rec.severity == Severity.CRITICAL: penalty += 15
    elif rec.severity == Severity.HIGH:   penalty += 8
    elif rec.severity == Severity.MEDIUM: penalty += 3
    elif rec.severity == Severity.LOW:    penalty += 1
health_score = max(0, min(100, 100 - penalty))

# Severity counts
severity_counts = {s.value: sum(1 for r in all_recs if r.severity == s) for s in Severity}
category_counts = {c.value: sum(1 for r in all_recs if r.category == c) for c in Category}

# Quick wins
quick_wins = [r for r in ranked if r['recommendation'].effort == Effort.QUICK_WIN][:5]

# Resource summary
resource_summary = {}
resource_stats = data.get('resource_stats', [])
if resource_stats:
    cpu_vals = [(r.get('avg_cpu_percent') or r.get('avg_cpu', 0) or 0) for r in resource_stats]
    io_vals = [(r.get('avg_data_io_percent') or r.get('avg_io', 0) or 0) for r in resource_stats]
    mem_vals = [(r.get('avg_memory_usage_percent') or r.get('avg_memory', 0) or 0) for r in resource_stats]
    resource_summary = {
        'avg_cpu': round(sum(cpu_vals) / max(len(cpu_vals), 1), 1),
        'max_cpu': round(max(cpu_vals), 1),
        'avg_io': round(sum(io_vals) / max(len(io_vals), 1), 1),
        'max_io': round(max(io_vals), 1),
        'avg_memory': round(sum(mem_vals) / max(len(mem_vals), 1), 1),
        'max_memory': round(max(mem_vals), 1),
    }

print(f"\n{'='*60}")
print(f"EXECUTIVE SUMMARY")
print(f"{'='*60}")
print(f"Health Score: {health_score}/100")
print(f"Total Recommendations: {len(all_recs)}")
print(f"Severity Breakdown: {severity_counts}")
print(f"Category Breakdown: {category_counts}")
print(f"Resource Summary: {resource_summary}")
print(f"Quick Wins: {len(quick_wins)}")


## 📊 Interactive Visualizations


In [ ]:
# CPU/IO Trend Chart (using Databricks native display)
resource_stats_records = data.get('resource_stats', [])
if resource_stats_records:
    import pandas as pd
    
    df_res = pd.DataFrame(resource_stats_records)
    
    # Normalize column names
    cpu_col = 'avg_cpu_percent' if 'avg_cpu_percent' in df_res.columns else 'avg_cpu'
    io_col = 'avg_data_io_percent' if 'avg_data_io_percent' in df_res.columns else 'avg_io'
    mem_col = 'avg_memory_usage_percent' if 'avg_memory_usage_percent' in df_res.columns else 'avg_memory'
    time_col = 'end_time' if 'end_time' in df_res.columns else 'ingestion_time'
    
    display_df = df_res[[time_col, cpu_col, io_col, mem_col]].rename(columns={
        time_col: 'Time',
        cpu_col: 'CPU %',
        io_col: 'Data IO %',
        mem_col: 'Memory %',
    })
    
    spark_display = spark.createDataFrame(display_df)
    display(spark_display)
else:
    print("No resource stats available for visualization.")


In [ ]:
# Recommendations by Severity (bar chart)
import pandas as pd

sev_df = pd.DataFrame([
    {"Severity": s.value, "Count": severity_counts.get(s.value, 0)}
    for s in Severity
])
display(spark.createDataFrame(sev_df))


In [ ]:
# Top 10 Recommendations Table
import pandas as pd

top_recs = []
for r in ranked[:10]:
    rec = r['recommendation']
    top_recs.append({
        'Rank': r['priority_rank'],
        'Score': r['priority_score'],
        'Severity': rec.severity.value,
        'Category': rec.category.value,
        'Title': rec.title,
        'Effort': rec.effort.value,
        'Action SQL': rec.action_sql[:100] if rec.action_sql else '',
    })

display(spark.createDataFrame(pd.DataFrame(top_recs)))


## 🗄️ Persist Recommendations to Storage Account (Parquet)

Persist all findings to `abfss://.../recommendations/{server}/{database}/year=YYYY/month=MM/day=DD/` as date-partitioned Parquet files.


In [ ]:
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# Build output DataFrame
output_rows = []
for r in ranked:
    rec = r['recommendation']
    output_rows.append({
        'priority_rank': r['priority_rank'],
        'priority_score': r['priority_score'],
        'severity': rec.severity.value,
        'category': rec.category.value,
        'title': rec.title,
        'description': rec.description,
        'impact_description': rec.impact_description,
        'action_sql': rec.action_sql,
        'effort': rec.effort.value,
        'risk': rec.risk.value,
        'confidence': rec.confidence.value,
        'estimated_impact_pct': rec.estimated_impact_pct,
        'source_metric': rec.source_metric,
        'health_score': health_score,
    })

if output_rows:
    df_out = spark.createDataFrame(pd.DataFrame(output_rows))
    df_out = (df_out
              .withColumn("server_name", lit(server_name))
              .withColumn("database_name", lit(database_name))
              .withColumn("analysis_timestamp", current_timestamp())
              .withColumn("lookback_days", lit(lookback_days))
    )
    
    year = now.strftime("%Y")
    month = now.strftime("%m")
    day = now.strftime("%d")
    recs_output_path = f"{RECS_BASE}/year={year}/month={month}/day={day}"
    
    df_out.write.mode("append").parquet(recs_output_path)
    print(f"✅ Wrote {len(output_rows)} recommendations to Parquet: {recs_output_path}")
else:
    recs_output_path = ""
    print("No recommendations to persist.")


## 📄 Generate HTML Report


In [ ]:
def generate_html_report(ranked, health_score, severity_counts, category_counts,
                          resource_summary, server_name, database_name, quick_wins):
    """Generate a standalone HTML executive report."""
    
    # Health bar color
    if health_score >= 80: bar_color = '#22c55e'
    elif health_score >= 60: bar_color = '#eab308'
    else: bar_color = '#ef4444'
    
    severity_colors = {'Critical': '#dc2626', 'High': '#ea580c', 'Medium': '#eab308', 'Low': '#22c55e', 'Info': '#3b82f6'}
    
    # Build recommendation rows
    rec_rows = ''
    for r in ranked:
        rec = r['recommendation']
        sev_color = severity_colors.get(rec.severity.value, '#6b7280')
        sql_block = f'<pre style="background:#1e293b;color:#e2e8f0;padding:8px;border-radius:4px;font-size:12px;overflow-x:auto;">{rec.action_sql}</pre>' if rec.action_sql else ''
        rec_rows += f'''
        <tr>
            <td>{r["priority_rank"]}</td>
            <td><strong>{r["priority_score"]}</strong></td>
            <td><span style="background:{sev_color};color:white;padding:2px 8px;border-radius:4px;font-size:12px;">{rec.severity.value}</span></td>
            <td>{rec.category.value}</td>
            <td><strong>{rec.title}</strong><br><small>{rec.description}</small></td>
            <td>{rec.effort.value}</td>
            <td>{sql_block}</td>
        </tr>'''
    
    # Severity badge bar
    sev_badges = ' '.join(
        f'<span style="background:{severity_colors.get(s,"#6b7280")};color:white;padding:4px 12px;border-radius:4px;margin:2px;">{s}: {c}</span>'
        for s, c in severity_counts.items() if c > 0
    )
    
    # Quick wins list
    qw_html = ''
    for qw in quick_wins:
        rec = qw['recommendation']
        qw_html += f'<li><strong>{rec.title}</strong> — {rec.description}</li>'
    
    html = f'''<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>Azure SQL Advisor Report — {server_name}/{database_name}</title>
    <style>
        body {{ font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #0f172a; color: #e2e8f0; margin: 0; padding: 20px; }}
        .container {{ max-width: 1200px; margin: 0 auto; }}
        h1 {{ color: #38bdf8; border-bottom: 2px solid #1e3a5f; padding-bottom: 12px; }}
        h2 {{ color: #7dd3fc; margin-top: 32px; }}
        .health-bar {{ background: #1e293b; border-radius: 12px; height: 32px; overflow: hidden; margin: 16px 0; }}
        .health-fill {{ height: 100%; border-radius: 12px; display: flex; align-items: center; justify-content: center; font-weight: bold; color: white; }}
        .summary-grid {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 16px; margin: 16px 0; }}
        .summary-card {{ background: #1e293b; padding: 16px; border-radius: 8px; text-align: center; }}
        .summary-card .value {{ font-size: 28px; font-weight: bold; color: #38bdf8; }}
        .summary-card .label {{ font-size: 13px; color: #94a3b8; margin-top: 4px; }}
        table {{ width: 100%; border-collapse: collapse; margin-top: 16px; }}
        th {{ background: #1e293b; padding: 12px; text-align: left; font-size: 13px; color: #94a3b8; text-transform: uppercase; }}
        td {{ padding: 12px; border-bottom: 1px solid #334155; vertical-align: top; }}
        tr:hover {{ background: #1e293b; }}
    </style>
</head>
<body>
<div class="container">
    <h1>🚀 Azure SQL Advisor Report</h1>
    <p><strong>Server:</strong> {server_name} | <strong>Database:</strong> {database_name} | <strong>Generated:</strong> {now.strftime("%Y-%m-%d %H:%M UTC")}</p>

    <h2>Health Score</h2>
    <div class="health-bar"><div class="health-fill" style="width:{health_score}%;background:{bar_color};">{health_score}/100</div></div>

    <div class="summary-grid">
        <div class="summary-card"><div class="value">{len(ranked)}</div><div class="label">Total Recommendations</div></div>
        <div class="summary-card"><div class="value">{resource_summary.get("avg_cpu", "N/A")}</div><div class="label">Avg CPU %</div></div>
        <div class="summary-card"><div class="value">{resource_summary.get("avg_io", "N/A")}</div><div class="label">Avg IO %</div></div>
        <div class="summary-card"><div class="value">{resource_summary.get("avg_memory", "N/A")}</div><div class="label">Avg Memory %</div></div>
    </div>

    <h2>Severity Breakdown</h2>
    <p>{sev_badges}</p>

    <h2>⚡ Quick Wins</h2>
    <ul>{qw_html if qw_html else '<li>No quick wins identified.</li>'}</ul>

    <h2>All Recommendations</h2>
    <table>
        <tr><th>#</th><th>Score</th><th>Severity</th><th>Category</th><th>Recommendation</th><th>Effort</th><th>Action SQL</th></tr>
        {rec_rows}
    </table>
</div>
</body>
</html>'''
    return html


# Generate and save report
report_html = generate_html_report(
    ranked, health_score, severity_counts, category_counts,
    resource_summary, server_name, database_name, quick_wins
)

# Save to DBFS / Storage
report_path = f"{ABFSS_BASE}/reports/{server_name}/{database_name}/azure_sql_advisor_report_{now.strftime('%Y%m%d_%H%M%S')}.html"
import tempfile
local_tmp = tempfile.mktemp(suffix=".html")
with open(local_tmp, "w") as f:
    f.write(report_html)
dbutils.fs.cp(f"file:{local_tmp}", report_path)
os.remove(local_tmp)

print(f"✅ HTML report saved to: {report_path}")

# Display inline
displayHTML(report_html)


In [ ]:
# Notebook exit value for orchestration
dbutils.notebook.exit(json.dumps({
    "status": "success",
    "health_score": health_score,
    "total_recommendations": len(all_recs),
    "severity_counts": severity_counts,
    "recommendations_path": recs_output_path,
    "report_path": report_path,
    "run_timestamp": now.isoformat(),
}))
